In [ ]:
"""
test.py — Textgenerierung mit zwei eigenen Modellen.

1) Pretrained-Modell aus Lab 5 (from-scratch, GPT-2-small Architektur, ctx 512)
2) Instruction-finetuntes gpt2-medium (355M) aus Lab 7 (ctx 1024)

Fuer beide Modelle werden dieselben Prompts mit identischen
Sampling-Einstellungen (top_k=25, temperature=1.4, 100 neue Tokens) generiert.
"""

import os
import sys

import torch
import tiktoken
from gpt import GPTModel, generate, text_to_token_ids, token_ids_to_text

# --- utils (gpt.py liegt in Abgabe/utils) ---
def _find_repo():
    start = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    d = start
    while True:
        if os.path.isfile(os.path.join(d, "utils", "gpt.py")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError(f"'Abgabe/utils/gpt.py' nicht gefunden ab {start}")
        d = parent

REPO = _find_repo()
sys.path.append(os.path.join(REPO, "utils"))

# --- Pfade zu den Modellen ---
PRETRAINED_PATH = os.path.join(
    REPO, "datasets", "pre-training",
    "model_pretrained_r100000_ctx512_e7_val1.44_20260616_074303.pth",
)
SFT_PATH = os.path.join(
    REPO, "datasets", "instruction_finetuning",
    "gpt2-medium355M-sft_own.pth",
)

# --- Device ---
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using {device} device.")

tokenizer = tiktoken.get_encoding("gpt2")

# --- Architektur-Configs ---
# Pretrained (from-scratch) – muss exakt zum gespeicherten Modell passen
PRETRAINED_CONFIG = {
    "vocab_size": 50257,
    "context_length": 512,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

# gpt2-medium (355M) fuer das SFT-Modell aus Lab 7
SFT_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1024,
    "n_heads": 16,
    "n_layers": 24,
    "drop_rate": 0.0,
    "qkv_bias": True,
}


def load_model(path, config):
    """Laedt ein GPTModel. Unterstuetzt sowohl rohe state_dicts als auch
    Checkpoint-Dicts mit 'model_state_dict' (+ optional 'config')."""
    ckpt = torch.load(path, map_location=device)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        if ckpt.get("config"):
            config = ckpt["config"]
        state_dict = ckpt["model_state_dict"]
    else:
        state_dict = ckpt
    model = GPTModel(config)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model, config


# --- Prompts (identisch fuer beide Modelle) ---
contexts = [
    "How do I make No-Bake Nut Cookies?"
]


def run(model, config, title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    for context in contexts:
        torch.manual_seed(123)
        token_ids = generate(
            model=model,
            idx=text_to_token_ids(context, tokenizer).to(device),
            max_new_tokens=1000,
            context_size=config["context_length"],
            top_k=25,
            temperature=1.4,
        )
        print(f"\nStart: {context}")
        print(f"Generated: {token_ids_to_text(token_ids, tokenizer)}")


if __name__ == "__main__":
    pretrained, pre_cfg = load_model(PRETRAINED_PATH, PRETRAINED_CONFIG)
    run(pretrained, pre_cfg, "1) Pretrained-Modell (Lab 5, from-scratch)")

    sft, sft_cfg = load_model(SFT_PATH, SFT_CONFIG)
    run(sft, sft_cfg, "2) Instruction-finetuntes gpt2-medium (Lab 7)")

Using mps device.

1) Pretrained-Modell (Lab 5, from-scratch)
